# Setup

Install the necessary dependencies and PILE dataset, from which we generate the corpus for the first k checkpoints as well as its corresponding index.

In [ ]:
import os

# Change the current working directory to the "team-aasa" folder
try:
    os.chdir("team-aasa")
    print(f"Current working directory changed to: {os.getcwd()}")
except FileNotFoundError:
    print("Error: The 'team-aasa' folder was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
! pip install -e .
! pip install nnsight

Some example code, downloading one shard of the Pile and building an index over it.

In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

local_dir = "./pile_deduped_shards"
snapshot_download(
    repo_id="EleutherAI/pile-deduped-pythia-preshuffled",
    repo_type="dataset",
    allow_patterns=["document-00000-of-00020.bin"],
    local_dir=local_dir,
    local_dir_use_symlinks=False,
)

shard0 = f"{local_dir}/document-00000-of-00020.bin"

In [ ]:
import numpy as np
from transformers import AutoTokenizer

PYTHIA_TOKENS_PER_STEP = 2_097_152  # 2^21
EARLY_STEPS_12 = np.array([0,1,2,4,8,16,32,64,128,256,512,1000], dtype=np.int64)

EARLY_CUTOFFS_12 = EARLY_STEPS_12 * np.int64(PYTHIA_TOKENS_PER_STEP)
T_MAX = int(EARLY_CUTOFFS_12.max())

tok = AutoTokenizer.from_pretrained("EleutherAI/pythia-70m-deduped")
dtype = np.uint16 if tok.vocab_size <= 2**16 else np.uint32
vocab_param = 2**16 if tok.vocab_size <= 2**16 else 2**32

In [ ]:
from pathlib import Path

prefix_dir = Path("./pile_prefix")
prefix_dir.mkdir(parents=True, exist_ok=True)
prefix_bin = prefix_dir / f"document-prefix-{T_MAX}.bin"

if not prefix_bin.exists():
    n_bytes = Path(shard0).stat().st_size
    tokens_in_shard = n_bytes // np.dtype(dtype).itemsize
    if T_MAX > tokens_in_shard:
        raise ValueError(f"Need {T_MAX:,} tokens but shard0 has only {tokens_in_shard:,}. "
                         "Download the next shard or lower MAX step.")
    src = np.memmap(shard0, dtype=dtype, mode="r")
    chunk = 64_000_000  # tokens per chunk (~128MB for u16)
    with open(prefix_bin, "wb") as f:
        for start in range(0, T_MAX, chunk):
            end = min(T_MAX, start + chunk)
            f.write(src[start:end].tobytes())
    del src

In [ ]:
from ngrams.backends.tokengrams import TokengramsIndex
from pathlib import Path

# tkg_idx = shard0 + ".tkg.idx"
tkg_idx = prefix_bin.with_suffix(".tkg.idx")
tg = TokengramsIndex()
tg.build_index(
    corpus_path=str(prefix_bin),
    index_path=str(tkg_idx),
    vocab=vocab_param,
    verbose=True,
    load_only=Path(tkg_idx).exists(),  # skip rebuild if the index is already there
)

# N-Grams Bucketing

After building an index in a cluster and building an index over a dataset, we are importing a parquet file with the cumulative n-grams that were computed.

In [ ]:
import numpy as np
import pandas as pd
from transformers import AutoTokenizer

PARQUET_PATH = "./cumulative.parquet"
METRIC = "count_cum"         # "count_cum" / "per_million_cum"
Q = 8                        # quantile buckets
K_PER_BUCKET = 30             # sample size per bucket
SEED = 0                     # RNG seed for reproducibility

In [ ]:
df = pd.read_parquet(PARQUET_PATH)

# Sanity checks
required_cols = {"phrase", "step", "count_cum", "per_million_cum"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Parquet missing required columns: {missing}")

steps = sorted(df["step"].unique().tolist())
last_step = steps[-1]
df_last = (df.loc[df["step"] == last_step, ["phrase", METRIC]]
             .copy()
             .assign(final_metric=lambda x: pd.to_numeric(x[METRIC], errors="coerce").fillna(0.0))
             .drop(columns=[METRIC]))

tok = AutoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", use_fast=True)
def tok_len(s: str) -> int:
    return len(tok.encode(s, add_special_tokens=False))
df_last["n_tokens"] = df_last["phrase"].map(tok_len)

In [ ]:
# --- within-length quantile bucketing on final_metric ---
def add_quantile_bucket_per_length(df_in: pd.DataFrame, q=4) -> pd.DataFrame:
    # rank within each n_tokens group to make qcut robust to ties
    df_tmp = df_in.copy()
    df_tmp["rank_in_len"] = df_tmp.groupby("n_tokens")["final_metric"].rank(method="average")
    # qcut per length; duplicates="drop" avoids empty bins when many ties
    df_tmp["bucket"] = (
        df_tmp.groupby("n_tokens", group_keys=False)["rank_in_len"]
              .apply(lambda s: pd.qcut(s, q=q, labels=False, duplicates="drop"))
    )
    return df_tmp.drop(columns=["rank_in_len"])

df_last = add_quantile_bucket_per_length(df_last, q=Q)

In [ ]:
# --- sample K per (n_tokens, bucket) ---
rng = np.random.default_rng(SEED)
samples = []
for nL, sub in df_last.groupby("n_tokens", sort=True):
    # how many distinct buckets actually exist for this length?
    present_buckets = sorted([int(b) for b in sub["bucket"].dropna().unique()])
    for b in present_buckets:
        block = sub[sub["bucket"] == b]
        if block.empty:
            continue
        take = min(K_PER_BUCKET, len(block))
        pick = block.sample(n=take, replace=False, random_state=int(rng.integers(0, 2**31)))
        samples.append(pick)

df_sampled = pd.concat(samples, ignore_index=True) if samples else df_last.iloc[0:0].copy()

In [ ]:
# --- bring along full time series for selected phrases (handy for plotting) ---
counts_w = df.pivot(index="phrase", columns="step", values="count_cum").reindex(columns=steps)
pm_w     = df.pivot(index="phrase", columns="step", values="per_million_cum").reindex(columns=steps)

In [ ]:
# materialize the tidy table you’ll keep
rows = []
for phrase, final_metric, n_tokens, bucket in df_sampled[["phrase","final_metric","n_tokens","bucket"]].itertuples(index=False):
    rows.append({
        "phrase": phrase,
        "n_tokens": int(n_tokens),
        "bucket": int(bucket),
        "final_metric": float(final_metric),
        "counts_cum": counts_w.loc[phrase, :].astype("int64").tolist(),
        "per_million_cum": pm_w.loc[phrase, :].astype("float64").tolist(),
    })
df_candidates = (pd.DataFrame(rows)
                 .sort_values(["n_tokens","bucket","final_metric"], ascending=[True, True, True])
                 .reset_index(drop=True))

print(f"Selected {len(df_candidates)} phrases across lengths {sorted(df_candidates['n_tokens'].unique())} "
      f"and buckets {sorted(df_candidates['bucket'].unique())}.")
display(df_candidates.head())

In [ ]:
# --------------------------- #
# Visualization: cumulative   #
# --------------------------- #
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- 0) Input sanity ---
try:
    df = df_candidates.copy()
except NameError:
    raise RuntimeError("df_candidates is not defined in this session.")

need = {"phrase","n_tokens","bucket","counts_cum","per_million_cum"}
missing = need - set(df.columns)
if missing:
    raise ValueError(f"df_candidates missing columns: {missing}")

# Convert counts_cum -> array [G, T]
C = np.asarray(df["counts_cum"].tolist(), dtype=np.int64)
G, T = C.shape

# Steps for x-axis:
# If you already have STEPS in scope (list of actual training steps), we’ll use it.
# Otherwise we’ll just use 0..T-1 as checkpoint indices.
try:
    STEPS  # noqa: F821
    steps = np.asarray(STEPS, dtype=np.int64)
    assert steps.shape[0] == T, "STEPS length must match number of columns in counts_cum"
except Exception:
    steps = np.arange(T, dtype=np.int64)

# Derived quantities
final_counts = C[:, -1]
df = df.assign(
    final_count=final_counts.astype(np.int64),
    bucket=lambda x: x["bucket"].astype(int),
    n_tokens=lambda x: x["n_tokens"].astype(int),
)

buckets = sorted(df["bucket"].unique().tolist())
lengths  = sorted(df["n_tokens"].unique().tolist())

# Utility: consistent save (optional)
def _save(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    print(f"Saved {path}")

# -------------- 1) ECDF + Boxplot of FINAL cumulative counts by bucket --------------
def plot_freq_distributions(df, out_prefix=None, logx=True):
    # ECDF (overall)
    fig, ax = plt.subplots(figsize=(7.5, 5.0))
    for b in buckets:
        x = df.loc[df["bucket"] == b, "final_count"].to_numpy(np.int64)
        if x.size == 0:
            continue
        x_sorted = np.sort(x)
        y = np.arange(1, x.size + 1) / x.size
        ax.step(x_sorted, y, where="post", label=f"bucket={b}", alpha=0.95)
    ax.set_xlabel("final cumulative count")
    ax.set_ylabel("ECDF")
    if logx:
        ax.set_xscale("log")
    ax.legend(title="bucket", fontsize=9)
    ax.set_title("ECDF of final cumulative counts by bucket (all lengths)")
    if out_prefix: _save(fig, f"{out_prefix}_ecdf_all.png")

    # Boxplot by bucket (overall)
    fig2, ax2 = plt.subplots(figsize=(7.5, 5.0))
    data = [df.loc[df["bucket"] == b, "final_count"].to_numpy(np.int64) for b in buckets]
    # If log, show log10(x+1)
    if logx:
        data_log = [np.log10(d.astype(float) + 1.0) for d in data]
        ax2.boxplot(data_log, labels=[str(b) for b in buckets], showfliers=False)
        ax2.set_ylabel("log10(final_count + 1)")
    else:
        ax2.boxplot(data, labels=[str(b) for b in buckets], showfliers=False)
        ax2.set_ylabel("final_count")
    ax2.set_xlabel("bucket")
    ax2.set_title("Final cumulative counts by bucket (all lengths)")
    if out_prefix: _save(fig2, f"{out_prefix}_box_all.png")

    # Per length: ECDF small multiples
    rows = len(lengths)
    fig3, axes = plt.subplots(rows, 1, figsize=(7.5, 3.5*rows), sharex=True)
    if rows == 1: axes = [axes]
    for ax, L in zip(axes, lengths):
        dL = df[df["n_tokens"] == L]
        for b in buckets:
            x = dL.loc[dL["bucket"] == b, "final_count"].to_numpy(np.int64)
            if x.size == 0:
                continue
            xs = np.sort(x)
            y = np.arange(1, xs.size + 1) / xs.size
            ax.step(xs, y, where="post", label=f"b={b}", alpha=0.95)
        ax.set_ylabel(f"ECDF (len={L})")
        if logx: ax.set_xscale("log")
        ax.legend(fontsize=8, ncols=min(4, len(buckets)))
    axes[-1].set_xlabel("final cumulative count")
    fig3.suptitle("ECDF of final cumulative counts by bucket, stratified by n-gram length")
    if out_prefix: _save(fig3, f"{out_prefix}_ecdf_by_len.png")

# -------------- 2) Rank–frequency (Zipf-like), overall + by bucket --------------
def plot_rank_frequency(df, out_prefix=None):
    # Overall
    f = df["final_count"].to_numpy(np.int64)
    f = f[f > 0]
    if f.size:
        ranks = np.arange(1, f.size + 1)
        f_sorted = np.sort(f)[::-1]
        fig, ax = plt.subplots(figsize=(7.0, 5.0))
        ax.plot(ranks, f_sorted, lw=1.5)
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("rank (log)")
        ax.set_ylabel("final cumulative count (log)")
        ax.set_title("Rank–frequency (all grams)")
        if out_prefix: _save(fig, f"{out_prefix}_rankfreq_all.png")

    # By bucket
    fig2, ax2 = plt.subplots(figsize=(7.0, 5.0))
    has_any = False
    for b in buckets:
        fb = df.loc[df["bucket"] == b, "final_count"].to_numpy(np.int64)
        fb = fb[fb > 0]
        if fb.size == 0:
            continue
        has_any = True
        rb = np.arange(1, fb.size + 1)
        fb_sorted = np.sort(fb)[::-1]
        ax2.plot(rb, fb_sorted, lw=1.2, label=f"bucket={b}")
    if has_any:
        ax2.set_xscale("log"); ax2.set_yscale("log")
        ax2.set_xlabel("rank (log)")
        ax2.set_ylabel("final cumulative count (log)")
        ax2.set_title("Rank–frequency by bucket")
        ax2.legend(fontsize=9)
        if out_prefix: _save(fig2, f"{out_prefix}_rankfreq_by_bucket.png")

# -------------- 3) Trajectories + per-bucket medians (optionally stratified by length) --------------
def plot_trajectories(C, steps, df, out_prefix=None, sample_per_bucket=12, alpha_lines=0.25, by_length=True):
    rng = np.random.default_rng(0)

    if by_length:
        # Facet by length rows
        rows = len(lengths)
        fig, axes = plt.subplots(rows, 1, figsize=(8.6, 3.6*rows), sharex=True)
        if rows == 1: axes = [axes]
        for ax, L in zip(axes, lengths):
            idx_L = df.index[df["n_tokens"] == L].to_numpy()
            if idx_L.size == 0:
                continue
            # sampled lines per bucket within this length
            for b in buckets:
                idx = df.index[(df["n_tokens"] == L) & (df["bucket"] == b)].to_numpy()
                if idx.size == 0:
                    continue
                if idx.size > sample_per_bucket:
                    idx = rng.choice(idx, size=sample_per_bucket, replace=False)
                for j in idx:
                    ax.plot(steps, C[j], lw=0.8, alpha=alpha_lines)
            # per-bucket median
            for b in buckets:
                idx = df.index[(df["n_tokens"] == L) & (df["bucket"] == b)].to_numpy()
                if idx.size == 0:
                    continue
                med = np.median(C[idx], axis=0)
                ax.plot(steps, med, lw=2.2, label=f"median b={b}")
            ax.set_ylabel(f"count (len={L})")
        axes[-1].set_xlabel("training step" if steps[0] != 0 or steps[-1] > T else "checkpoint index")
        fig.suptitle("Cumulative frequency trajectories (sampled) + per-bucket medians")
        if out_prefix: _save(fig, f"{out_prefix}_traj_by_len.png")
    else:
        fig, ax = plt.subplots(figsize=(8.6, 5.0))
        # sampled lines overall per bucket
        for b in buckets:
            idx = df.index[df["bucket"] == b].to_numpy()
            if idx.size == 0:
                continue
            take = min(sample_per_bucket, idx.size)
            for j in np.random.choice(idx, size=take, replace=False):
                ax.plot(steps, C[j], lw=0.8, alpha=alpha_lines)
        # medians per bucket
        for b in buckets:
            idx = df.index[df["bucket"] == b].to_numpy()
            if idx.size == 0: continue
            med = np.median(C[idx], axis=0)
            ax.plot(steps, med, lw=2.2, label=f"median b={b}")
        ax.set_xlabel("training step" if steps[0] != 0 or steps[-1] > T else "checkpoint index")
        ax.set_ylabel("cumulative count")
        ax.set_title("Cumulative frequency trajectories (sampled) + per-bucket medians")
        ax.legend(fontsize=9)
        if out_prefix: _save(fig, f"{out_prefix}_traj_overall.png")

# -------------- 4) Heatmap of log cumulative counts (sorted by length, bucket, first-seen) --------------
def plot_heatmap(C, steps, df, out_prefix=None):
    mask = C > 0
    first_seen_idx = np.where(mask.any(axis=1), mask.argmax(axis=1), -1)  # -1 if never seen

    # Sorting key: length asc, bucket asc, first-seen (with -1 at the end)
    fs_sort = np.where(first_seen_idx < 0, C.shape[1] + 10**9, first_seen_idx)
    order = np.lexsort((fs_sort, df["bucket"].to_numpy(), df["n_tokens"].to_numpy()))
    C_sorted = C[order]
    buckets_sorted = df["bucket"].to_numpy()[order]
    lengths_sorted = df["n_tokens"].to_numpy()[order]

    H = np.log1p(C_sorted.astype(np.float64))

    fig, ax = plt.subplots(figsize=(9.2, 7.2))
    im = ax.imshow(H, aspect="auto", interpolation="nearest", origin="upper")
    ax.set_xlabel("training step" if steps[0] != 0 or steps[-1] > T else "checkpoint index")
    ax.set_ylabel("ngrams (sorted by length, bucket, first-seen)")
    ax.set_title("log(1 + cumulative counts)")
    cbar = fig.colorbar(im, ax=ax); cbar.set_label("log(1 + count)")

    # draw separators where length or bucket changes
    for col_name, arr in [("n_tokens", lengths_sorted), ("bucket", buckets_sorted)]:
        change_rows = np.nonzero(np.r_[True, np.diff(arr) != 0])[0]  # first of each block
        # draw lines above each block start (skip the very first row)
        for r in change_rows[1:]:
            ax.axhline(r - 0.5, color="white", lw=0.6, alpha=0.9)
    if out_prefix: _save(fig, f"{out_prefix}_heatmap.png")

# -------------- 6) Convenience: render all --------------
def render_all(out_dir=None):
    out_prefix = None
    if out_dir:
        import os
        os.makedirs(out_dir, exist_ok=True)
        out_prefix = f"{out_dir.rstrip('/')}/cum"
    plot_freq_distributions(df, out_prefix=out_prefix, logx=True)
    plot_rank_frequency(df, out_prefix=out_prefix)
    plot_trajectories(C, steps, df, out_prefix=out_prefix, sample_per_bucket=12, by_length=True)
    plot_heatmap(C, steps, df, out_prefix=out_prefix)
    # plot_first_seen_hist(C, steps, out_prefix=out_prefix)

# ---- run everything (optionally set an output dir to save PNGs) ----
render_all(out_dir=None)


# JSD Visualization

This pipeline assumes access to parquet file(s) with the data from the Neural Embeddings pipeline.

In [ ]:
# ============================================================
# End-to-end: JSD between n-gram bins via Pythia activations
# + Per-neuron JSD contributions × Polysemanticity analyses
# + Frequency-affinity (sum-weighted) & Mean-affinity (mass-normalized)
# + Coverage-only analyses and diagnostics
# + Visualizations (global + per-step grid)
# + Layerwise heatmaps, deciles, partial regressions, effect sizes,
#   stability bars, and links to JSD
# Runs for multiple models (Pythia-70M, Pythia-160M)
# ============================================================

from __future__ import annotations
from typing import List, Dict, Tuple, Optional
import numpy as np
import pandas as pd
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from nnsight import LanguageModel
from scipy.stats import spearmanr, mannwhitneyu, kruskal, zscore

# ---------- Plot style ----------
mpl.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300, "font.size": 12,
    "axes.titlesize": 14, "axes.labelsize": 12, "legend.fontsize": 10,
    "xtick.labelsize": 10, "ytick.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
})
from cycler import cycler
mpl.rcParams["axes.prop_cycle"] = cycler(color=mpl.cm.get_cmap("tab10").colors)

# ---------- Models ----------
MODELS = [
    {
        "tag": "pythia-70m",
        "title": "Pythia-70M",
        "model_id": "EleutherAI/pythia-70m-deduped",
        "steps": [1000, 13000, 23000, 33000, 43000, 53000,
                  63000, 73000, 83000, 93000, 103000, 113000,
                  123000, 133000, 143000],
        "layers": list(range(0, 6)),
        "clusters_parquet": "./pythia70m_embeddings.parquet",
    },
    {
        "tag": "pythia-160m",
        "title": "Pythia-160M",
        "model_id": "EleutherAI/pythia-160m-deduped",
        "steps": [1000, 13000, 23000, 33000, 43000, 53000,
                  63000, 73000, 83000, 93000, 103000, 113000,
                  123000, 133000, 143000],
        "layers": list(range(0, 12)),
        "clusters_parquet": "./pythia160m_embeddings.parquet",
    },
]

# ---------- Config ----------
BUCKETS_TO_COMPARE = [0, 7]
BATCH_SIZE = 32
DTYPE = torch.float16
DEVICE_MAP = "auto"
TEMPLATE_PREFIX = "This is a place in the world:"
TEMPLATE_SUFFIX = "."
COUNTS_PARQUET = "./cumulative.parquet"   # columns: phrase, step, per_million_cum (or count_cum)
OUTROOT = Path("./runs_jsd_poly"); OUTROOT.mkdir(parents=True, exist_ok=True)
EPS = 1e-12

# ---------- Inputs (expects df_candidates in memory) ----------
try:
    df_candidates  # noqa
except NameError:
    raise RuntimeError("df_candidates not found. It must include: phrase, n_tokens, bucket.")
present_buckets = sorted(df_candidates["bucket"].dropna().unique().astype(int).tolist())
use_buckets = [b for b in BUCKETS_TO_COMPARE if b in present_buckets]
if len(use_buckets) < 2:
    print(f"[warn] Requested buckets {BUCKETS_TO_COMPARE}, present={present_buckets}")

bucket2phrases: Dict[int, List[str]] = {
    b: sorted(df_candidates.loc[df_candidates["bucket"] == b, "phrase"].unique().tolist())
    for b in use_buckets
}

# ---------- Load cumulative counts ----------
df_counts_all = pd.read_parquet(COUNTS_PARQUET)
if not {"phrase", "step"}.issubset(df_counts_all.columns):
    raise ValueError(f"{COUNTS_PARQUET} needs at least ['phrase','step', <count column>].")
count_col = "per_million_cum" if "per_million_cum" in df_counts_all.columns else ("count_cum" if "count_cum" in df_counts_all.columns else None)
if count_col is None:
    raise ValueError(f"{COUNTS_PARQUET} must have 'per_million_cum' or 'count_cum'.")
df_counts_all = df_counts_all[df_counts_all["phrase"].isin(
    np.unique([p for b in use_buckets for p in bucket2phrases[b]])
)].copy()

# ---------- Helpers ----------
def build_text(phrase: str) -> str:
    return f"{TEMPLATE_PREFIX}{phrase}{TEMPLATE_SUFFIX}"

def _anchor_indices_for_batch(lm: LanguageModel, phrases: List[str]) -> np.ndarray:
    tok = lm.tokenizer
    Lsuf = len(tok.encode(TEMPLATE_SUFFIX, add_special_tokens=False))
    anchors = []
    for p in phrases:
        ids = tok.encode(build_text(p), add_special_tokens=False)
        if Lsuf >= len(ids): raise ValueError("Template issue.")
        anchors.append(len(ids) - Lsuf - 1)
    return np.asarray(anchors, dtype=int)

def capture_postmlp_at_anchor(lm: LanguageModel, texts: List[str], anchors: np.ndarray, layers: List[int]) -> Dict[int, np.ndarray]:
    with lm.trace(texts) as tr:
        caps = {L: lm.gpt_neox.layers[L].mlp.dense_4h_to_h.input.save() for L in layers}
        _ = lm.output.save()
    out: Dict[int, np.ndarray] = {}
    for L in layers:
        v = caps[L].value.to("cpu").float().detach().numpy()     # [B,S,H]
        out[L] = v[np.arange(v.shape[0]), anchors, :]            # [B,H]
    return out

def relu_probs(A: np.ndarray, eps: float = EPS) -> np.ndarray:
    X = np.maximum(A, 0.0)
    denom = X.sum(axis=1, keepdims=True)
    P = np.divide(X, np.maximum(denom, eps), out=np.zeros_like(X), where=(denom > 0))
    return P

def jsd_per_dim(P: np.ndarray, Q: np.ndarray, eps: float = EPS) -> np.ndarray:
    P = np.asarray(P, np.float64); P = P / (P.sum() + eps)
    Q = np.asarray(Q, np.float64); Q = Q / (Q.sum() + eps)
    M = 0.5 * (P + Q)
    return 0.5 * (P * (np.log(P + eps) - np.log(M + eps)) + Q * (np.log(Q + eps) - np.log(M + eps)))

_rng = np.random.default_rng(0)
def bootstrap_spearman_ci(x, y, n_boot=1000, alpha=0.05):
    x = np.asarray(x, float); y = np.asarray(y, float)
    if len(x) < 3: return np.nan, (np.nan, np.nan)
    vals = []
    for _ in range(n_boot):
        idx = _rng.integers(0, len(x), size=len(x))
        r, _ = spearmanr(x[idx], y[idx])
        vals.append(r if np.isfinite(r) else np.nan)
    lo = np.nanpercentile(vals, 2.5); hi = np.nanpercentile(vals, 97.5)
    return float(np.nanmean(vals)), (float(lo), float(hi))

def fdr_bh(pvals: np.ndarray, alpha: float = 0.05):
    p = np.asarray(pvals, float)
    if p.size == 0: return np.array([], bool), np.array([], float)
    order = np.argsort(p); ranked = p[order]
    thresh = alpha * (np.arange(1, p.size+1) / p.size)
    passed = ranked <= np.maximum.accumulate(thresh)
    mask = np.zeros_like(passed, bool); mask[order] = passed
    adj = np.empty_like(p); adj[order] = np.minimum.accumulate((p.size/np.arange(p.size, 0, -1)) * ranked[::-1])[::-1]
    return mask, np.clip(adj, 0, 1)

def cliffs_delta(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    if a.size == 0 or b.size == 0: return np.nan
    A = np.sort(a); B = np.sort(b); i=j=more=less=0; na=len(A); nb=len(B)
    while i<na and j<nb:
        if A[i] > B[j]: more += (na-i); j += 1
        elif A[i] < B[j]: less += (nb-j); i += 1
        else: i += 1; j += 1
    return float((more - less) / (na*nb))

def cohens_d(a, b, hedges_correction=True):
    a=np.asarray(a,float); b=np.asarray(b,float)
    if a.size<2 or b.size<2: return np.nan, np.nan
    sa2=a.var(ddof=1); sb2=b.var(ddof=1); n1=a.size; n2=b.size
    sp=np.sqrt(((n1-1)*sa2+(n2-1)*sb2)/max(1,(n1+n2-2)))
    d=(a.mean()-b.mean())/sp if np.isfinite(sp) and sp>0 else np.nan
    if hedges_correction and np.isfinite(d):
        J=1-3/(4*(n1+n2)-9); return d, J*d
    return d, d

def bootstrap_ci(func, a, b, n_boot=1000):
    a=np.asarray(a,float); b=np.asarray(b,float)
    if a.size==0 or b.size==0: return np.nan, (np.nan,np.nan)
    vals=[]
    for _ in range(n_boot):
        ai=_rng.integers(0,a.size,size=a.size); bi=_rng.integers(0,b.size,size=b.size)
        vals.append(func(a[ai], b[bi]))
    lo=np.nanpercentile(vals,2.5); hi=np.nanpercentile(vals,97.5)
    return float(np.nanmean(vals)), (float(lo), float(hi))

def savefig_dual(fig, path_png: Path, path_pdf: Optional[Path] = None):
    fig.savefig(path_png, dpi=300, bbox_inches="tight")
    if path_pdf is None: path_pdf = path_png.with_suffix(".pdf")
    fig.savefig(path_pdf, dpi=300, bbox_inches="tight")
    print(f"[saved] {path_png}\n[saved] {path_pdf}")

# ---------- Load clusters (polysemanticity) ----------
def load_clusters_from_export_parquet(path: str) -> pd.DataFrame:
    dfw = pd.read_parquet(path)
    if not {"layer","neuron"}.issubset(dfw.columns): raise ValueError("Parquet must contain layer, neuron.")
    idxs = [i for i in range(1000) if f"checkpoint_{i}_step" in dfw.columns and f"checkpoint_{i}_num_clusters" in dfw.columns]
    if not idxs: raise ValueError("No checkpoint_{i}_step/num_clusters columns found.")
    base = dfw[["layer","neuron"]].astype(int); recs=[]
    for i in idxs:
        steps = pd.to_numeric(dfw[f"checkpoint_{i}_step"], errors="coerce")
        ncl   = pd.to_numeric(dfw[f"checkpoint_{i}_num_clusters"], errors="coerce")
        part = pd.DataFrame({"layer": base["layer"], "neuron": base["neuron"], "step": steps.astype("Int64"), "n_clusters": ncl}).dropna(subset=["step"])
        part["step"]=part["step"].astype(int); recs.append(part)
    return pd.concat(recs, ignore_index=True).sort_values(["layer","neuron","step"]).reset_index(drop=True)

# ---------- Sweep: JSD, affinity, mean-affinity, firing mass ----------
def sweep_bins_jsd_with_contrib_and_affinity(
    model_id: str, model_title: str, steps: List[int], layers: List[int],
    bucket_phrases: Dict[int, List[str]], df_counts_all: pd.DataFrame, count_col: str, batch_size: int = 32,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[Tuple[int,int,int], np.ndarray], Dict[Tuple[int,int], np.ndarray], Dict[Tuple[int,int], np.ndarray], Dict[Tuple[int,int], np.ndarray]]:
    """
    Returns:
      df_jsd, df_contrib,
      avgP: {(step,layer,bucket)->[H]},
      aff_sum: {(step,layer)->[H]} = Σ_p P(h|p) * log f(p)
      aff_mean: {(step,layer)->[H]} = aff_sum / Σ_p P(h|p)           (mean log-frequency)
      fire_mass: {(step,layer)->[H]} = Σ_p P(h|p)                    (neuron activation mass)
    """
    jsd_rows, contrib_rows = [], []
    avgP: Dict[Tuple[int,int,int], np.ndarray] = {}
    aff_sum: Dict[Tuple[int,int], np.ndarray] = {}
    aff_mean: Dict[Tuple[int,int], np.ndarray] = {}
    fire_mass: Dict[Tuple[int,int], np.ndarray] = {}

    for s in steps:
        rev = f"step{s}"
        print(f"[info] loading {model_title} @ {rev}")
        lm = LanguageModel(model_id, revision=rev, device_map=DEVICE_MAP, torch_dtype=DTYPE, dispatch=True)

        bucket_texts, bucket_anchors, bucket_logf = {}, {}, {}
        for b, plist in bucket_phrases.items():
            bucket_texts[b] = [build_text(p) for p in plist]
            bucket_anchors[b] = _anchor_indices_for_batch(lm, plist)
            dfc = (df_counts_all[df_counts_all["step"] == s]
                   .set_index("phrase")[count_col]
                   .reindex(plist).fillna(0.0))
            bucket_logf[b] = np.log(np.clip(dfc.to_numpy(float), 1e-12, None))

        for L in layers:
            H_aff_sum = None
            H_mass = None

            for b, plist in bucket_phrases.items():
                if not plist:
                    avgP[(s,L,b)] = np.array([]); continue
                rows=[]
                texts=bucket_texts[b]; anchors=bucket_anchors[b]
                for i0 in range(0, len(plist), batch_size):
                    sl = slice(i0, i0+batch_size)
                    caps = capture_postmlp_at_anchor(lm, texts[sl], anchors[sl], [L])
                    P = relu_probs(caps[L], eps=EPS)   # [B,H]
                    rows.append(P)
                P_b = np.vstack(rows) if rows else np.zeros((0,0), float)
                if P_b.size:
                    meanP = P_b.mean(axis=0); meanP = meanP / (meanP.sum() + EPS)
                    avgP[(s,L,b)] = meanP
                    contrib_aff = P_b.T @ bucket_logf[b]                  # [H]
                    mass = P_b.sum(axis=0)                                 # [H]
                    H_aff_sum = contrib_aff if H_aff_sum is None else (H_aff_sum + contrib_aff)
                    H_mass = mass if H_mass is None else (H_mass + mass)
                else:
                    avgP[(s,L,b)] = np.array([])

            if H_mass is None:
                aff_sum[(s,L)] = np.array([]); aff_mean[(s,L)] = np.array([]); fire_mass[(s,L)] = np.array([])
            else:
                aff_sum[(s,L)] = H_aff_sum
                fire_mass[(s,L)] = H_mass
                aff_mean[(s,L)] = np.divide(H_aff_sum, np.maximum(H_mass, 1e-20), out=np.zeros_like(H_aff_sum), where=(H_mass>0))

            # JSD (first two buckets)
            if len(use_buckets) >= 2:
                b0,b1 = use_buckets[:2]
                P0 = avgP.get((s,L,b0), np.array([])); P1 = avgP.get((s,L,b1), np.array([]))
                if P0.size and P1.size and P0.shape==P1.shape:
                    contrib = jsd_per_dim(P0, P1, eps=EPS)
                    jsd_rows.append({"step": s, "layer": L, "bucket_a": b0, "bucket_b": b1, "JSD": float(contrib.sum())})
                    for i, c in enumerate(contrib):
                        contrib_rows.append({"step": s, "layer": L, "neuron": i, "jsd_contrib": float(c)})
                else:
                    jsd_rows.append({"step": s, "layer": L, "bucket_a": b0, "bucket_b": b1, "JSD": np.nan})

        del lm
        torch.cuda.empty_cache()

    df_jsd = pd.DataFrame(jsd_rows).sort_values(["layer","step"]).reset_index(drop=True)
    df_contrib = pd.DataFrame(contrib_rows).sort_values(["layer","step","neuron"]).reset_index(drop=True)
    return df_jsd, df_contrib, avgP, aff_sum, aff_mean, fire_mass

# ---------- Core JSD plots ----------
def plot_jsd_over_steps(df_jsd, outdir: Path, model_title: str, tag: str):
    layers = sorted(df_jsd["layer"].dropna().unique().astype(int))
    fig, ax = plt.subplots(figsize=(9.5,5.2))
    for L in layers:
        d = df_jsd[df_jsd["layer"]==L]
        ax.plot(d["step"], d["JSD"], marker="o", lw=1.8, label=f"Layer {L}")
    ax.set_xlabel("Training step"); ax.set_ylabel("JSD(bin0, bin7)")
    ax.set_title(f"JSD between bin 0 and bin 7 across steps ({model_title})")
    ax.legend(ncols=min(4,len(layers))); savefig_dual(fig, outdir / f"{tag}_jsd_over_steps.png"); plt.close(fig)

def plot_avgprob_heatmaps(avgP, steps, layers, buckets, outdir: Path, model_title: str, tag: str):
    if len(buckets) < 2: return
    b0,b1=buckets[:2]
    for s in steps:
        for L in layers:
            P0=avgP.get((s,L,b0),np.array([])); P1=avgP.get((s,L,b1),np.array([]))
            if P0.size==0 or P1.size==0 or P0.shape[0]!=P1.shape[0]: continue
            M=np.vstack([P0,P1])
            fig,ax=plt.subplots(figsize=(11.0,3.0),constrained_layout=True)
            im=ax.imshow(M,aspect="auto",interpolation="nearest",origin="upper",cmap="viridis")
            ax.set_xlabel("Neuron index"); ax.set_yticks([0,1]); ax.set_yticklabels([f"bucket={b0}",f"bucket={b1}"])
            ax.set_title(f"Avg prob by neuron | step={s} | layer={L} ({model_title})")
            cbar=fig.colorbar(im,ax=ax); cbar.set_label("Probability")
            savefig_dual(fig, outdir / f"{tag}_avgprob_step{s}_layer{L}.png"); plt.close(fig)

# ---------- JSD × polysemanticity ----------
def corr_over_steps_with_ci(df, steps, outdir: Path, model_title: str, tag: str):
    rows=[]
    for s in steps:
        sub=df[df["step"]==s].dropna(subset=["jsd_contrib","n_clusters"])
        if len(sub)<3: rows.append({"step":s,"spearman":np.nan,"p":np.nan,"lo":np.nan,"hi":np.nan,"n":len(sub)}); continue
        r,p=spearmanr(sub["jsd_contrib"], sub["n_clusters"])
        _,(lo,hi)=bootstrap_spearman_ci(sub["jsd_contrib"], sub["n_clusters"])
        rows.append({"step":s,"spearman":float(r),"p":float(p),"lo":float(lo),"hi":float(hi),"n":len(sub)})
    d=pd.DataFrame(rows); sig,padj=fdr_bh(d["p"].fillna(1.0).values,0.05); d["p_adj"]=padj; d["sig_0.05_fdr"]=sig
    fig,ax=plt.subplots(figsize=(8.8,5.3))
    ax.plot(d["step"],d["spearman"],marker="o",lw=1.6,label="Spearman r")
    ax.fill_between(d["step"],d["lo"],d["hi"],alpha=0.22,label="95% bootstrap CI")
    for x,y,m in zip(d["step"],d["spearman"],d["sig_0.05_fdr"]):
        if m: ax.scatter([x],[y],s=60,marker="*",color="tab:red",zorder=3)
    ax.axhline(0,color="k",lw=0.8,alpha=0.6)
    ax.set_xlabel("Training step"); ax.set_ylabel("Spearman corr(JSD contribution, #clusters)")
    ax.set_title(f"Global correlation over training ({model_title})"); ax.legend(ncols=2)
    savefig_dual(fig, outdir / f"{tag}_corr_over_steps_ci.png"); plt.close(fig)
    return d

def stratified_stats_and_plot(df, step: int, outdir: Path, model_title: str, tag: str):
    sub=df[df["step"]==step].dropna(subset=["jsd_contrib","n_clusters"]).copy()
    if sub.empty: return None
    sub["poly_bin"]=pd.cut(sub["n_clusters"],bins=[0,2,5,10,100],labels=["low (1-2)","med (3-5)","high (6-10)","very high (10+)"])
    groups=[(n,g["jsd_contrib"].astype(float).values) for n,g in sub.groupby("poly_bin")]
    labels=[n for n,_ in groups]; data=[v for _,v in groups]
    H,p_kw=(kruskal(*data) if len(data)>=2 else (np.nan,np.nan))
    fig,ax=plt.subplots(figsize=(8.6,5.2)); ax.boxplot(data,labels=labels,showfliers=False)
    meds=[np.median(d) if len(d) else np.nan for d in data]
    ax.plot(np.arange(1,len(meds)+1),meds,marker="o",lw=0,color="tab:red",label="median")
    ax.set_xlabel("Polysemanticity bin (#clusters)"); ax.set_ylabel("JSD contribution")
    ax.set_title(f"Stratified JSD by polysemanticity (step={step}, {model_title})\nKruskal–Wallis H={H:.3g}, p={p_kw:.3g}")
    ax.legend(loc="upper left"); savefig_dual(fig, outdir / f"{tag}_stratified_step{step}.png"); plt.close(fig)
    # pairwise
    pairs=[]
    for i in range(len(data)):
        for j in range(i+1,len(data)):
            a,b=data[i],data[j]
            if len(a)==0 or len(b)==0: continue
            stat,p=mannwhitneyu(a,b,alternative="two-sided"); delta=cliffs_delta(a,b)
            pairs.append((labels[i],labels[j],float(p),float(stat),float(delta)))
    if pairs:
        pvals=np.array([p for _,_,p,_,_ in pairs],float); rej,padj=fdr_bh(pvals,0.05)
        df_pairs=pd.DataFrame([{"step":step,"group_a":li,"group_b":lj,"p_raw":p,"p_fdr":pa,"reject_fdr_0.05":bool(r),"U_stat":st,"cliffs_delta":de}
                               for (li,lj,p,st,de),r,pa in zip(pairs,rej,padj)])
    else:
        df_pairs=pd.DataFrame()
    return {"kw_H":H,"kw_p":p_kw,"pairs":df_pairs}

# ---------- Affinity tables ----------
def build_affinity_tables(aff_sum, aff_mean, fire_mass) -> pd.DataFrame:
    rows=[]
    for (s,L), v in aff_sum.items():
        if v is None or len(v)==0: continue
        ms = aff_mean.get((s,L), np.zeros_like(v))
        fm = fire_mass.get((s,L), np.zeros_like(v))
        for h,(a,am,f) in enumerate(zip(v,ms,fm)):
            rows.append({"step":s,"layer":L,"neuron":h,"freq_affinity":float(a),"mean_affinity":float(am),"firing_mass":float(f)})
    return pd.DataFrame(rows)

# ---------- Affinity × polysemanticity over steps ----------
def corr_affinity_poly_over_steps(df_aff_merge, steps, outdir: Path, model_title: str, tag: str, col="freq_affinity", label="freq-affinity"):
    rows=[]
    for s in steps:
        sub=df_aff_merge[df_aff_merge["step"]==s].dropna(subset=[col,"n_clusters"])
        if len(sub)<3: rows.append({"step":s,"spearman":np.nan,"p":np.nan,"lo":np.nan,"hi":np.nan}); continue
        r,p=spearmanr(sub[col], sub["n_clusters"]); _,(lo,hi)=bootstrap_spearman_ci(sub[col], sub["n_clusters"])
        rows.append({"step":s,"spearman":float(r),"p":float(p),"lo":float(lo),"hi":float(hi)})
    d=pd.DataFrame(rows); rej,padj=fdr_bh(d["p"].fillna(1.0).to_numpy(),0.05); d["p_fdr"]=padj; d["sig_0.05_fdr"]=rej
    fig,ax=plt.subplots(figsize=(8.8,5.0))
    ax.plot(d["step"],d["spearman"],marker="o",lw=1.6,label="Spearman r"); ax.fill_between(d["step"],d["lo"],d["hi"],alpha=0.22,label="95% CI")
    for x,y,m in zip(d["step"],d["spearman"],d["sig_0.05_fdr"]):
        if m: ax.scatter([x],[y],s=60,marker="*",color="tab:red",zorder=3)
    ax.axhline(0,color="k",lw=0.8,alpha=0.6)
    ax.set_xlabel("Training step"); ax.set_ylabel(f"Spearman corr({label}, #clusters)")
    ax.set_title(f"{label} × polysemanticity over training ({model_title})"); ax.legend(ncols=2)
    savefig_dual(fig, outdir / f"{tag}_{col}_poly_corr_over_steps.png"); plt.close(fig)
    return d

def affinity_bin_tests(df_merge_contrib, steps, outdir: Path, model_title: str, tag: str, col="freq_affinity"):
    rows=[]
    for s in steps:
        sub=df_merge_contrib[df_merge_contrib["step"]==s].dropna(subset=[col,"jsd_contrib"])
        if len(sub)<10: continue
        q25,q75=np.nanpercentile(sub[col],[25,75])
        low=sub[sub[col]<=q25]["jsd_contrib"].to_numpy(float); high=sub[sub[col]>=q75]["jsd_contrib"].to_numpy(float)
        if low.size==0 or high.size==0: continue
        U,p=mannwhitneyu(high,low,alternative="two-sided"); delta=cliffs_delta(high,low)
        _,(dlo,dhi)=bootstrap_ci(cliffs_delta,high,low,1000)
        _,g=cohens_d(high,low,hedges_correction=True)
        def _g(a,b): _,gg=cohens_d(a,b,hedges_correction=True); return gg
        _,(glo,ghi)=bootstrap_ci(_g,high,low,1000)
        rows.append({"step":s,"n_low":int(low.size),"n_high":int(high.size),"U":float(U),"p":float(p),
                     "cliffs_delta":float(delta),"delta_lo":float(dlo),"delta_hi":float(dhi),"hedges_g":float(g),
                     "g_lo":float(glo),"g_hi":float(ghi)})
    d=pd.DataFrame(rows)
    if d.empty: return d
    rej,padj=fdr_bh(d["p"].values,0.05); d["p_fdr"]=padj; d["reject_fdr_0.05"]=rej
    # plots
    fig,ax=plt.subplots(figsize=(9.0,5.0))
    ax.plot(d["step"],d["hedges_g"],marker="o"); ax.fill_between(d["step"],d["g_lo"],d["g_hi"],alpha=0.2)
    ax.scatter(d["step"][d["reject_fdr_0.05"]], d["hedges_g"][d["reject_fdr_0.05"]], color="tab:red", marker="*", s=70, zorder=3)
    ax.axhline(0,color="k",lw=0.8,alpha=0.6); ax.set_xlabel("Training step"); ax.set_ylabel("Hedges' g (high − low)")
    ax.set_title(f"Effect size: high vs low {col} (JSD contribution)\n{model_title}")
    savefig_dual(fig, outdir / f"{tag}_{col}_effectsize_g_high_vs_low.png"); plt.close(fig)
    fig,ax=plt.subplots(figsize=(9.0,4.8))
    ax.plot(d["step"],d["cliffs_delta"],marker="o"); ax.axhline(0,color="k",lw=0.8,alpha=0.6)
    ax.set_xlabel("Training step"); ax.set_ylabel("Cliff's δ (high vs low)")
    ax.set_title(f"Rank-biserial effect: high vs low {col}\n{model_title}")
    savefig_dual(fig, outdir / f"{tag}_{col}_effectsize_delta_high_vs_low.png"); plt.close(fig)
    return d

# ---------- Scatter visuals ----------
def _pick_rep_steps(steps: List[int], max_panels: int = 8) -> List[int]:
    if len(steps)<=max_panels: return steps
    idx=np.linspace(0,len(steps)-1,num=max_panels,dtype=int); return [steps[i] for i in idx]

def scatter_grid_affinity_vs_poly(df_aff_merge, steps, outdir: Path, model_title: str, tag: str, col="freq_affinity",
                                  max_panels: int=8, per_panel_cap: int=40000, use_hexbin_threshold: int=15000):
    sel_steps=_pick_rep_steps(steps,max_panels=max_panels); n=len(sel_steps); ncols=min(4,n); nrows=int(np.ceil(n/ncols))
    fig,axes=plt.subplots(nrows,ncols,figsize=(4.0*ncols+0.5,3.5*nrows+0.5),constrained_layout=True); axes=np.array(axes).reshape(-1)
    for ax,s in zip(axes,sel_steps):
        sub=df_aff_merge[df_aff_merge["step"]==s].dropna(subset=[col,"n_clusters"])
        if sub.empty: ax.set_axis_off(); continue
        if len(sub)>per_panel_cap: sub=sub.sample(per_panel_cap,random_state=0)
        x=sub[col].to_numpy(float); y=sub["n_clusters"].to_numpy(float)
        if len(sub)>=use_hexbin_threshold:
            hb=ax.hexbin(x,y,gridsize=50,cmap="viridis",mincnt=1); cb=fig.colorbar(hb,ax=ax); cb.set_label("count")
        else:
            ax.scatter(x,y,s=6,alpha=0.6)
        if len(sub)>=3:
            r,p=spearmanr(x,y); ax.set_title(f"step={s}  ρ={r:.2f}, p={p:.2g}")
        else:
            ax.set_title(f"step={s}")
        ax.set_xlabel(f"{col.replace('_',' ')}"); ax.set_ylabel("#clusters")
    for i in range(len(sel_steps),len(axes)): axes[i].set_axis_off()
    fig.suptitle(f"Polysemanticity vs {col.replace('_',' ')} across checkpoints ({model_title})",y=1.02)
    savefig_dual(fig, outdir / f"{tag}_{col}_vs_poly_scatter_grid.png"); plt.close(fig)

def scatter_global_step_colored(df_aff_merge, steps, outdir: Path, model_title: str, tag: str, col="freq_affinity", max_points: int=300_000):
    sub=df_aff_merge.dropna(subset=[col,"n_clusters","step"]).copy()
    if sub.empty: return
    step_min,step_max=float(min(steps)),float(max(steps)); sub["step_norm"]=(sub["step"]-step_min)/max(1e-9,(step_max-step_min))
    if len(sub)>max_points: sub=sub.sample(max_points,random_state=0)
    fig,ax=plt.subplots(figsize=(8.8,6.0))
    sc=ax.scatter(sub[col].to_numpy(float), sub["n_clusters"].to_numpy(float), c=sub["step_norm"].to_numpy(float), cmap="viridis", s=6, alpha=0.6, linewidths=0)
    cbar=fig.colorbar(sc,ax=ax); cbar.set_label("normalized training step")
    ax.set_xlabel(f"{col.replace('_',' ')}"); ax.set_ylabel("#clusters")
    ax.set_title(f"Polysemanticity vs {col.replace('_',' ')} (all checkpoints colored by step)\n{model_title}")
    savefig_dual(fig, outdir / f"{tag}_{col}_vs_poly_scatter_global.png"); plt.close(fig)

# ========================== NEW: The 6 layerwise analyses ==========================

# (1) Per-layer correlation heatmap (affinity × polysemanticity)
def affinity_poly_layerwise(df_aff_merge, steps, outdir: Path, model_title: str, tag: str, col="freq_affinity"):
    rows=[]; layers=sorted(df_aff_merge["layer"].dropna().unique().astype(int))
    for s in steps:
        for L in layers:
            sub=df_aff_merge[(df_aff_merge.step==s)&(df_aff_merge.layer==L)].dropna(subset=[col,"n_clusters"])
            if len(sub)<3: rows.append({"step":s,"layer":L,"spearman":np.nan,"p":np.nan,"n":len(sub)}); continue
            r,p=spearmanr(sub[col], sub["n_clusters"]); rows.append({"step":s,"layer":L,"spearman":float(r),"p":float(p),"n":len(sub)})
    d=pd.DataFrame(rows)
    # heatmap with FDR per step
    fig,ax=plt.subplots(figsize=(10.0,4.8))
    layers_sorted=sorted(d["layer"].unique()); steps_sorted=steps
    M=np.full((len(layers_sorted),len(steps_sorted)),np.nan,float); Sig=np.zeros_like(M,bool)
    for j,s in enumerate(steps_sorted):
        dj=d[d["step"]==s]
        pvals=dj.set_index("layer").reindex(layers_sorted)["p"].fillna(1.0).values
        rej,_=fdr_bh(pvals,0.05); Sig[:,j]=rej
        M[:,j]=dj.set_index("layer").reindex(layers_sorted)["spearman"].values
    im=ax.imshow(M,aspect="auto",origin="lower",extent=[steps_sorted[0],steps_sorted[-1],layers_sorted[0]-0.5,layers_sorted[-1]+0.5],cmap="coolwarm",vmin=-0.3,vmax=0.3)
    ax.set_yticks(layers_sorted); ax.set_xlabel("Training step"); ax.set_ylabel("Layer")
    ax.set_title(f"Layer×Step Spearman r: {col.replace('_',' ')} vs #clusters ({model_title})")
    cbar=plt.colorbar(im,ax=ax); cbar.set_label("Spearman r")
    for i,L in enumerate(layers_sorted):
        for j,s in enumerate(steps_sorted):
            if Sig[i,j]: ax.plot(s,L,marker="o",markersize=3.5,color="white")
    savefig_dual(fig, outdir / f"{tag}_{col}_layerwise_corr_heatmap.png"); plt.close(fig)
    return d

# (2) Per-layer decile curves: mean clusters vs affinity decile
def affinity_deciles_per_layer(df_aff_merge, steps, layers, outdir: Path, model_title: str, tag: str, col="freq_affinity"):
    sel_steps=_pick_rep_steps(steps, max_panels=4)
    for s in sel_steps:
        for L in layers:
            sub=df_aff_merge[(df_aff_merge.step==s)&(df_aff_merge.layer==L)].dropna(subset=[col,"n_clusters"])
            if len(sub)<50: continue
            sub=sub.copy()
            sub["decile"]=pd.qcut(sub[col], 10, labels=False, duplicates="drop")
            g=sub.groupby("decile")["n_clusters"]
            x=g.mean().index.astype(int); y=g.mean().values
            se=g.std().values/np.sqrt(np.maximum(1,g.size().values))
            fig,ax=plt.subplots(figsize=(5.2,3.4))
            ax.errorbar(x,y,yerr=se,marker="o",lw=1.2)
            ax.set_xlabel(f"{col.replace('_',' ')} decile (low → high)"); ax.set_ylabel("mean #clusters")
            ax.set_title(f"{model_title} • L{L} • step {s}")
            savefig_dual(fig, outdir / f"{tag}_{col}_deciles_L{L}_S{s}.png"); plt.close(fig)

# (3) Partial regressions within layer (controls)
def layerwise_regressions(df_aff_merge, steps, outdir: Path, model_title: str, tag: str, col="freq_affinity"):
    rows=[]
    for s in steps:
        sub=df_aff_merge[df_aff_merge["step"]==s].dropna(subset=[col,"n_clusters","firing_mass"])
        if sub.empty: continue
        for L, g in sub.groupby("layer"):
            if len(g)<20: continue
            X=np.column_stack([zscore(g[col].to_numpy(float)), zscore(g["firing_mass"].replace(0,np.nan).to_numpy(float))])
            y=g["n_clusters"].to_numpy(float)
            X=np.nan_to_num(X, nan=0.0)
            XtX=X.T@X + 1e-8*np.eye(X.shape[1])
            beta=np.linalg.solve(XtX, X.T@y)
            r1,_=spearmanr(g[col], g["n_clusters"])
            rows.append({"step":int(s),"layer":int(L),"beta_affinity":float(beta[0]),"beta_mass":float(beta[1]),"spearman_raw":float(r1),"n":int(len(g))})
    d=pd.DataFrame(rows).sort_values(["layer","step"])
    d.to_csv(outdir / f"{tag}_{col}_layerwise_regressions.csv", index=False)
    # beta heatmap
    if not d.empty:
        layers=sorted(d["layer"].unique()); steps_sorted=sorted(d["step"].unique())
        M=np.full((len(layers),len(steps_sorted)),np.nan,float)
        for i,L in enumerate(layers):
            for j,s in enumerate(steps_sorted):
                sub=d[(d.layer==L)&(d.step==s)]
                if not sub.empty: M[i,j]=sub.iloc[0]["beta_affinity"]
        fig,ax=plt.subplots(figsize=(10.0,4.8))
        im=ax.imshow(M,aspect="auto",origin="lower",extent=[steps_sorted[0],steps_sorted[-1],layers[0]-0.5,layers[-1]+0.5],cmap="coolwarm")
        ax.set_yticks(layers); ax.set_xlabel("Training step"); ax.set_ylabel("Layer")
        ax.set_title(f"OLS coefficient of {col.replace('_',' ')} predicting #clusters (controls: firing mass)\n{model_title}")
        cbar=plt.colorbar(im,ax=ax); cbar.set_label("β (std units)")
        savefig_dual(fig, outdir / f"{tag}_{col}_layerwise_beta_heatmap.png"); plt.close(fig)
    return d

# (4) Effect sizes by layer (within-layer high vs low affinity → #clusters)
def effect_sizes_by_layer(df_aff_merge, steps, outdir: Path, model_title: str, tag: str, col="freq_affinity"):
    rows=[]
    for s in steps:
        for L, g in df_aff_merge[df_aff_merge["step"]==s].groupby("layer"):
            sub=g.dropna(subset=[col,"n_clusters"])
            if len(sub)<20: continue
            q25,q75=np.nanpercentile(sub[col],[25,75])
            low=sub[sub[col]<=q25]["n_clusters"].to_numpy(float); high=sub[sub[col]>=q75]["n_clusters"].to_numpy(float)
            if low.size<2 or high.size<2: continue
            _,gval=cohens_d(high,low,hedges_correction=True)
            delta=cliffs_delta(high,low)
            rows.append({"step":int(s),"layer":int(L),"hedges_g":float(gval),"cliffs_delta":float(delta),"n_low":int(low.size),"n_high":int(high.size)})
    d=pd.DataFrame(rows).sort_values(["layer","step"])
    d.to_csv(outdir / f"{tag}_{col}_layerwise_effectsizes.csv", index=False)
    if not d.empty:
        fig,ax=plt.subplots(figsize=(9.6,5.2))
        for L in sorted(d["layer"].unique()):
            dl=d[d["layer"]==L]
            ax.plot(dl["step"], dl["hedges_g"], marker="o", lw=1.2, label=f"L{L}")
        ax.axhline(0,color="k",lw=0.8,alpha=0.6)
        ax.set_xlabel("Training step"); ax.set_ylabel("Hedges' g (high − low)")
        ax.set_title(f"Within-layer effect size: high vs low {col.replace('_',' ')} on #clusters\n{model_title}")
        ax.legend(ncols=min(5, d["layer"].nunique()))
        savefig_dual(fig, outdir / f"{tag}_{col}_layerwise_effectsizes_lines.png"); plt.close(fig)
    return d

# (5) Stability across steps: per-layer median ρ with bootstrap CI
def stability_bars(df_aff_merge, steps, outdir: Path, model_title: str, tag: str, col="freq_affinity"):
    rows=[]
    for L, g in df_aff_merge.groupby("layer"):
        vals=[]
        for s in steps:
            sub=g[g["step"]==s].dropna(subset=[col,"n_clusters"])
            if len(sub)<3: continue
            r,_=spearmanr(sub[col], sub["n_clusters"]); vals.append(r)
        if len(vals)==0: continue
        vals=np.array(vals,float)
        boots=[]
        for _ in range(1000):
            idx=_rng.integers(0,len(vals),size=len(vals)); boots.append(np.nanmedian(vals[idx]))
        lo=np.nanpercentile(boots,2.5); hi=np.nanpercentile(boots,97.5)
        rows.append({"layer":int(L),"median_r":float(np.nanmedian(vals)),"lo":float(lo),"hi":float(hi),"n_steps":int(len(vals))})
    d=pd.DataFrame(rows).sort_values("layer")
    fig,ax=plt.subplots(figsize=(8.8,4.8))
    ax.errorbar(d["layer"], d["median_r"], yerr=[d["median_r"]-d["lo"], d["hi"]-d["median_r"]], fmt="o", lw=1.2, capsize=3)
    ax.axhline(0,color="k",lw=0.8,alpha=0.6)
    ax.set_xlabel("Layer"); ax.set_ylabel("Median Spearman r across steps")
    ax.set_title(f"Stability of {col.replace('_',' ')}–polysemanticity association by layer\n{model_title}")
    savefig_dual(fig, outdir / f"{tag}_{col}_stability_bars.png"); plt.close(fig)
    return d

# (6) Link to JSD: correlations & simple mediation-style regressions
def link_affinity_jsd_poly(df_aff_merge_contrib, steps, outdir: Path, model_title: str, tag: str, col="freq_affinity"):
    rows=[]
    for s in steps:
        for L, g in df_aff_merge_contrib[df_aff_merge_contrib["step"]==s].groupby("layer"):
            sub=g.dropna(subset=[col,"n_clusters","jsd_contrib","firing_mass"])
            if len(sub)<20: continue
            r1,_=spearmanr(sub[col], sub["jsd_contrib"])          # affinity → jsd
            r2,_=spearmanr(sub["n_clusters"], sub["jsd_contrib"]) # poly → jsd
            r3,_=spearmanr(sub["firing_mass"], sub["jsd_contrib"])# coverage → jsd
            # OLS (std) with both predictors
            X=np.column_stack([
                zscore(sub[col].to_numpy(float)),
                zscore(sub["n_clusters"].to_numpy(float)),
                zscore(sub["firing_mass"].replace(0,np.nan).to_numpy(float))
            ])
            y=zscore(sub["jsd_contrib"].to_numpy(float))
            X=np.nan_to_num(X, nan=0.0)
            XtX=X.T@X + 1e-8*np.eye(3); beta=np.linalg.solve(XtX, X.T@y)
            rows.append({"step":int(s),"layer":int(L),
                         "rho_aff_jsd":float(r1),"rho_poly_jsd":float(r2),"rho_mass_jsd":float(r3),
                         "beta_affinity":float(beta[0]),"beta_poly":float(beta[1]),"beta_mass":float(beta[2]),
                         "n":int(len(sub))})
    d=pd.DataFrame(rows).sort_values(["layer","step"])
    d.to_csv(outdir / f"{tag}_{col}_link_jsd_poly.csv", index=False)
    # heatmaps of betas
    if not d.empty:
        for name in ["beta_affinity","beta_poly","beta_mass"]:
            layers=sorted(d["layer"].unique()); steps_sorted=sorted(d["step"].unique())
            M=np.full((len(layers),len(steps_sorted)),np.nan,float)
            for i,L in enumerate(layers):
                for j,s in enumerate(steps_sorted):
                    sub=d[(d.layer==L)&(d.step==s)]
                    if not sub.empty: M[i,j]=sub.iloc[0][name]
            fig,ax=plt.subplots(figsize=(10.0,4.8))
            im=ax.imshow(M,aspect="auto",origin="lower",extent=[steps_sorted[0],steps_sorted[-1],layers[0]-0.5,layers[-1]+0.5],cmap="coolwarm")
            ax.set_yticks(layers); ax.set_xlabel("Training step"); ax.set_ylabel("Layer")
            ax.set_title(f"{name} predicting JSD contribution (std units)\n{model_title}")
            cbar=plt.colorbar(im,ax=ax); cbar.set_label(name)
            savefig_dual(fig, outdir / f"{tag}_{col}_{name}_heatmap.png"); plt.close(fig)
    return d

# ========================== NEW: Coverage-only analyses & diagnostics ==========================

def add_log_transforms(df: pd.DataFrame) -> pd.DataFrame:
    df=df.copy()
    if "firing_mass" in df.columns:
        df["firing_mass_log1p"]=np.log1p(df["firing_mass"].clip(lower=0))
    return df

def corr_coverage_poly_over_steps(df_aff_merge, steps, outdir: Path, model_title: str, tag: str, col="firing_mass", label="coverage (firing mass)"):
    return corr_affinity_poly_over_steps(df_aff_merge, steps, outdir, model_title, tag, col=col, label=label)

def scatter_coverage_maps(df_aff_merge, steps, outdir: Path, model_title: str, tag: str):
    # heavy-tailed view
    scatter_grid_affinity_vs_poly(df_aff_merge, steps, outdir, model_title, tag, col="firing_mass_log1p")
    scatter_global_step_colored(df_aff_merge, steps, outdir, model_title, tag, col="firing_mass_log1p")

def partial_spearman(x, y, z) -> float:
    """
    Approximate partial Spearman by ranking then correlating Pearson residuals.
    """
    def _rank(v):
        v=np.asarray(v,float)
        r=pd.Series(v).rank(method="average").to_numpy()
        return (r - r.mean()) / (r.std(ddof=0) + 1e-12)
    xr, yr, zr = _rank(x), _rank(y), _rank(z)
    # linear residualization
    Z = np.column_stack([np.ones_like(zr), zr])
    beta_x = np.linalg.lstsq(Z, xr, rcond=None)[0]
    beta_y = np.linalg.lstsq(Z, yr, rcond=None)[0]
    rx = xr - Z @ beta_x
    ry = yr - Z @ beta_y
    # Pearson on residuals
    if rx.std(ddof=1) < 1e-12 or ry.std(ddof=1) < 1e-12: return np.nan
    return float(np.corrcoef(rx, ry)[0,1])

def partial_corrs_over_steps(df, steps, outdir: Path, model_title: str, tag: str):
    rows=[]
    for s in steps:
        sub=df[df["step"]==s].dropna(subset=["freq_affinity","firing_mass","n_clusters"])
        if len(sub)<10:
            rows.append({"step":s,"rho_A_C_given_S":np.nan,"rho_S_C_given_A":np.nan,"n":len(sub)})
            continue
        rA = partial_spearman(sub["freq_affinity"], sub["n_clusters"], sub["firing_mass"])
        rS = partial_spearman(sub["firing_mass"], sub["n_clusters"], sub["freq_affinity"])
        rows.append({"step":s,"rho_A_C_given_S":rA,"rho_S_C_given_A":rS,"n":len(sub)})
    d=pd.DataFrame(rows)
    # plot
    fig,ax=plt.subplots(figsize=(9.0,5.0))
    ax.plot(d["step"], d["rho_A_C_given_S"], marker="o", label="partial: affinity | coverage")
    ax.plot(d["step"], d["rho_S_C_given_A"], marker="s", label="partial: coverage | affinity")
    ax.axhline(0,color="k",lw=0.8,alpha=0.6)
    ax.set_xlabel("Training step"); ax.set_ylabel("partial Spearman r")
    ax.set_title(f"Partial correlations with #clusters ({model_title})")
    ax.legend()
    savefig_dual(fig, outdir / f"{tag}_partial_corrs_over_steps.png"); plt.close(fig)
    d.to_csv(outdir / f"{tag}_partial_corrs_over_steps.csv", index=False)
    return d

def coverage_matched_pairs(df, steps, outdir: Path, model_title: str, tag: str, n_bins: int = 10):
    """
    Within each step, bin by coverage deciles, then compare #clusters across high vs low freq_affinity quartiles.
    """
    rows=[]
    for s in steps:
        sub=df[df["step"]==s].dropna(subset=["freq_affinity","firing_mass","n_clusters"])
        if len(sub)<100: continue
        sub=sub.copy()
        sub["cov_dec"]=pd.qcut(sub["firing_mass"], q=n_bins, labels=False, duplicates="drop")
        for dcv, g in sub.groupby("cov_dec"):
            if len(g)<40: continue
            q25,q75=np.nanpercentile(g["freq_affinity"], [25,75])
            lo=g[g["freq_affinity"]<=q25]["n_clusters"].to_numpy(float)
            hi=g[g["freq_affinity"]>=q75]["n_clusters"].to_numpy(float)
            if lo.size<2 or hi.size<2: continue
            delta=cliffs_delta(hi,lo)
            _,gval=cohens_d(hi,lo,hedges_correction=True)
            rows.append({"step":int(s),"cov_dec":int(dcv),"cliffs_delta":float(delta),"hedges_g":float(gval),
                         "n_lo":int(lo.size),"n_hi":int(hi.size)})
    d=pd.DataFrame(rows)
    if d.empty: return d
    # summary plot
    fig,ax=plt.subplots(figsize=(9.0,5.0))
    for k,g in d.groupby("cov_dec"):
        ax.plot(g["step"], g["cliffs_delta"], marker="o", lw=1.0, label=f"cov decile {int(k)}")
    ax.axhline(0,color="k",lw=0.8,alpha=0.6)
    ax.set_xlabel("Training step"); ax.set_ylabel("Cliff's δ (high − low affinity)")
    ax.set_title(f"Coverage-matched effect on #clusters by decile ({model_title})")
    if d["cov_dec"].nunique()<=12: ax.legend(ncol=3)
    savefig_dual(fig, outdir / f"{tag}_coverage_matched_pairs_delta.png"); plt.close(fig)
    d.to_csv(outdir / f"{tag}_coverage_matched_pairs.csv", index=False)
    return d

def permutation_test_affinity_components(df, steps, outdir: Path, model_title: str, tag: str, n_perm: int = 200):
    """
    Shuffle mean-affinity across neurons (keep coverage fixed) and shuffle coverage (keep mean fixed),
    recompute sum-affinity = S * mu, and correlate with #clusters. Compare to baseline.
    """
    rows=[]
    for s in steps:
        sub=df[df["step"]==s].dropna(subset=["freq_affinity","mean_affinity","firing_mass","n_clusters"])
        if len(sub)<50: continue
        x_base=sub["freq_affinity"].to_numpy(float)
        y=sub["n_clusters"].to_numpy(float)
        r_base,_=spearmanr(x_base,y)
        mu=sub["mean_affinity"].to_numpy(float)
        S=sub["firing_mass"].to_numpy(float)
        # keep indices for within-step shuffles
        r_mu=[]; r_S=[]
        for _ in range(n_perm):
            mu_shuffled=np.random.permutation(mu)
            S_shuffled=np.random.permutation(S)
            A_mu = S * mu_shuffled
            A_S  = S_shuffled * mu
            r1,_=spearmanr(A_mu, y)
            r2,_=spearmanr(A_S , y)
            r_mu.append(r1); r_S.append(r2)
        rows.append({
            "step":int(s), "r_base":float(r_base),
            "r_mu_mean":float(np.nanmean(r_mu)), "r_mu_p_le":float(np.mean(np.array(r_mu) >= r_base)),
            "r_S_mean":float(np.nanmean(r_S)),   "r_S_p_le":float(np.mean(np.array(r_S)  >= r_base)),
            "n":int(len(sub)), "n_perm":int(n_perm)
        })
    d=pd.DataFrame(rows)
    # plot comparison
    if not d.empty:
        fig,ax=plt.subplots(figsize=(8.8,4.8))
        ax.plot(d["step"], d["r_base"], marker="o", label="baseline r(A,sum)")
        ax.plot(d["step"], d["r_mu_mean"], marker="s", label="shuffle μ (keep S)")
        ax.plot(d["step"], d["r_S_mean"], marker="^", label="shuffle S (keep μ)")
        ax.axhline(0,color="k",lw=0.8,alpha=0.6)
        ax.set_xlabel("Training step"); ax.set_ylabel("Spearman r with #clusters")
        ax.set_title(f"Permutation ablations of sum-affinity ({model_title})")
        ax.legend()
        savefig_dual(fig, outdir / f"{tag}_permutation_affinity_components.png"); plt.close(fig)
    d.to_csv(outdir / f"{tag}_permutation_affinity_components.csv", index=False)
    return d

# ============================================================
# MAIN
# ============================================================
for M in MODELS:
    tag, model_title, model_id = M["tag"], M["title"], M["model_id"]
    steps, layers, clusters_path = M["steps"], M["layers"], M["clusters_parquet"]

    OUTDIR = OUTROOT / tag; OUTDIR.mkdir(parents=True, exist_ok=True)
    OUTDIR_JSD = OUTDIR / "jsd";  OUTDIR_JSD.mkdir(parents=True, exist_ok=True)
    OUTDIR_POLY = OUTDIR / "poly"; OUTDIR_POLY.mkdir(parents=True, exist_ok=True)
    OUTDIR_FREQ = OUTDIR / "freq"; OUTDIR_FREQ.mkdir(parents=True, exist_ok=True)

    # 1) Sweep
    df_jsd, df_contrib, avgP, aff_sum, aff_mean, fire_mass = sweep_bins_jsd_with_contrib_and_affinity(
        model_id=model_id, model_title=model_title, steps=steps, layers=layers,
        bucket_phrases=bucket2phrases, df_counts_all=df_counts_all, count_col=count_col, batch_size=BATCH_SIZE
    )

    df_jsd.to_csv(OUTDIR_JSD / f"{tag}_between_bin_jsd.csv", index=False)
    df_contrib.to_csv(OUTDIR_JSD / f"{tag}_per_neuron_jsd_contrib.csv", index=False)
    plot_jsd_over_steps(df_jsd, OUTDIR_JSD, model_title, tag)
    plot_avgprob_heatmaps(avgP, steps, layers, use_buckets, OUTDIR_JSD, model_title, tag)

    # 2) Polysemanticity merge
    clusters_long = load_clusters_from_export_parquet(clusters_path)
    clusters_long = clusters_long[clusters_long["step"].isin(steps)].copy()

    # 3) Affinity tables (+ mean-affinity & firing mass) and merges
    df_aff_all = build_affinity_tables(aff_sum, aff_mean, fire_mass)
    df_aff_all.to_csv(OUTDIR_FREQ / f"{tag}_neuron_affinity_tables.csv", index=False)

    df_aff = df_aff_all[["step","layer","neuron","freq_affinity","mean_affinity","firing_mass"]]
    df_aff_merge = df_aff.merge(clusters_long, on=["step","layer","neuron"], how="left")
    df_aff_merge = add_log_transforms(df_aff_merge)
    df_aff_merge.to_csv(OUTDIR_FREQ / f"{tag}_affinity_merged_with_clusters.csv", index=False)

    # 4) Correlations over steps (sum-weighted + mean-affinity + coverage)
    df_aff_corr = corr_affinity_poly_over_steps(df_aff_merge, steps, OUTDIR_FREQ, model_title, tag, col="freq_affinity", label="freq-affinity")
    df_aff_corr.to_csv(OUTDIR_FREQ / f"{tag}_affinity_poly_corr_over_steps.csv", index=False)
    df_mean_corr = corr_affinity_poly_over_steps(df_aff_merge, steps, OUTDIR_FREQ, model_title, tag, col="mean_affinity", label="mean-affinity")
    df_mean_corr.to_csv(OUTDIR_FREQ / f"{tag}_MEANaffinity_poly_corr_over_steps.csv", index=False)
    df_cov_corr  = corr_coverage_poly_over_steps(df_aff_merge, steps, OUTDIR_FREQ, model_title, tag, col="firing_mass", label="coverage (firing mass)")
    df_cov_corr.to_csv(OUTDIR_FREQ / f"{tag}_COVERAGE_poly_corr_over_steps.csv", index=False)

    # 5) High vs Low affinity -> JSD contribution (sum-weighted affinity)
    df_aff_merge_with_contrib = df_aff_merge.merge(df_contrib, on=["step","layer","neuron"], how="left")
    df_aff_effects = affinity_bin_tests(df_aff_merge_with_contrib, steps, OUTDIR_FREQ, model_title, tag, col="freq_affinity")
    if not df_aff_effects.empty:
        df_aff_effects.to_csv(OUTDIR_FREQ / f"{tag}_affinity_high_vs_low_effects.csv", index=False)

    # 6) Scatter visuals (affinity & coverage)
    scatter_grid_affinity_vs_poly(df_aff_merge, steps, OUTDIR_FREQ, model_title, tag, col="freq_affinity")
    scatter_global_step_colored(df_aff_merge, steps, OUTDIR_FREQ, model_title, tag, col="freq_affinity")
    scatter_coverage_maps(df_aff_merge, steps, OUTDIR_FREQ, model_title, tag)

    # ======================= Layerwise analyses =======================
    d_heat = affinity_poly_layerwise(df_aff_merge, steps, OUTDIR_FREQ, model_title, tag, col="freq_affinity")
    d_heat.to_csv(OUTDIR_FREQ / f"{tag}_freq_affinity_layerwise_corr.csv", index=False)
    affinity_deciles_per_layer(df_aff_merge, steps, layers, OUTDIR_FREQ, model_title, tag, col="freq_affinity")
    d_reg = layerwise_regressions(df_aff_merge, steps, OUTDIR_FREQ, model_title, tag, col="freq_affinity")
    d_eff = effect_sizes_by_layer(df_aff_merge, steps, OUTDIR_FREQ, model_title, tag, col="freq_affinity")
    d_stab = stability_bars(df_aff_merge, steps, OUTDIR_FREQ, model_title, tag, col="freq_affinity")
    d_stab.to_csv(OUTDIR_FREQ / f"{tag}_freq_affinity_stability_bars.csv", index=False)

    # ======================= Coverage-only diagnostics =======================
    d_partial = partial_corrs_over_steps(df_aff_merge, steps, OUTDIR_FREQ, model_title, tag)
    d_match   = coverage_matched_pairs(df_aff_merge, steps, OUTDIR_FREQ, model_title, tag, n_bins=10)
    d_perm    = permutation_test_affinity_components(df_aff_merge, steps, OUTDIR_FREQ, model_title, tag, n_perm=200)

    # ======================= Link to JSD (affinity/poly/coverage → jsd_contrib) =======================
    d_link = link_affinity_jsd_poly(df_aff_merge_with_contrib, steps, OUTDIR_FREQ, model_title, tag, col="freq_affinity")

print("\n[done] All models processed. Outputs in:", OUTROOT.resolve())
